# Cloud-Based AI Data Analysis and Predictive Analytics System

**STTP:** Cloud-Based AI Solutions: From Data Processing to Generative Models

This notebook demonstrates a complete introductory machine-learning workflow in Google Colab: dataset preparation, exploratory data analysis (EDA), preprocessing, model training, evaluation, and prediction.

> The dataset is synthetic and created for educational demonstration. It does not represent real customers or operational business data.


## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score
)

RANDOM_STATE = 42


## 2. Create and store a sample dataset

In [ ]:
# Create a reproducible synthetic binary-classification dataset.
X, y = make_classification(
    n_samples=1000,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    n_classes=2,
    weights=[0.58, 0.42],
    class_sep=1.0,
    random_state=RANDOM_STATE
)

feature_names = [
    "usage_frequency", "service_duration", "support_requests",
    "monthly_spend", "engagement_score", "account_age",
    "response_time", "satisfaction_score"
]

df = pd.DataFrame(X, columns=feature_names)
df["retention_status"] = y
df.head()


In [ ]:
# Save a local CSV copy. In Colab, it can also be downloaded or saved to Drive/GCS.
csv_path = "sample_customer_retention.csv"
df.to_csv(csv_path, index=False)
print(f"Dataset saved as: {csv_path}")
print("Dataset shape:", df.shape)


### Optional cloud-storage step
In Google Colab, you can mount Google Drive and copy the CSV there. For Google Cloud Storage (GCS), create a bucket in your own GCP project and upload the CSV through the Cloud Console. Do not place passwords, service-account keys, or other credentials in a public notebook.


In [ ]:
# Optional: mount Google Drive in Google Colab
# from google.colab import drive
# drive.mount('/content/drive')
# df.to_csv('/content/drive/MyDrive/sample_customer_retention.csv', index=False)


## 3. Exploratory Data Analysis (EDA)

In [ ]:
display(df.describe())
print("\\nMissing values by column:")
display(df.isnull().sum())
print("\\nTarget distribution:")
display(df["retention_status"].value_counts())


In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="retention_status")
plt.title("Distribution of Retention Classes")
plt.xlabel("Retention status (0 = Class 0, 1 = Class 1)")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()


## 4. Data preprocessing and train-test split

In [ ]:
X = df.drop(columns=["retention_status"])
y = df["retention_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))


## 5. Train a machine-learning model

In [ ]:
# Scaling is included in a pipeline; the Random Forest model is trained on the training split.
model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)
print("Model training completed.")


## 6. Evaluate model performance

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy: {accuracy:.3f}")
print(f"ROC-AUC:  {auc:.3f}")
print("\\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["Class 0", "Class 1"], cmap="Blues"
)
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


## 7. Generate a sample prediction

In [ ]:
sample_record = X_test.iloc[[0]]
predicted_class = int(model.predict(sample_record)[0])
predicted_probability = float(model.predict_proba(sample_record)[0, 1])

print("Predicted class:", predicted_class)
print(f"Estimated probability for Class 1: {predicted_probability:.3f}")
display(sample_record)


## 8. Conclusion

This mini-project demonstrates the core stages of a cloud-ready AI workflow: data handling, EDA, preprocessing, model development, evaluation, and prediction. The notebook runs in Google Colab and can be extended by storing datasets in Google Drive or Google Cloud Storage and deploying the trained model through an API.

**Important:** Results are generated from synthetic data for learning purposes. They should not be interpreted as evidence of real-world business performance.
